<a href="https://colab.research.google.com/github/kgsledu-cell/ai-agent/blob/main/notebooks/06_gemini_video_news_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini Video News Workflow

Build a short news video with Gemini agents and Veo: search for news, refine it, write a script, create a video prompt, then generate and download an MP4.

## 1. Install dependencies

In [73]:
%pip install -qU openai-agents ddgs google-genai

## 2. Configure Gemini

Add `GEMINI_API_KEY` to Colab Secrets before running this cell. Video generation may take several minutes and requires Veo access for your Gemini API key.

In [74]:
from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

GEMINI_API_KEY = userdata.get("GeminiAPIKey")
GEMINI_MODEL_NAME = "gemini-3.5-flash"
VIDEO_MODEL_NAME = "veo-3.1-generate-preview"

if not GEMINI_API_KEY:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets before running this notebook.")

set_tracing_disabled(disabled=True)
gemini_model = OpenAIChatCompletionsModel(
    model=GEMINI_MODEL_NAME,
    openai_client=AsyncOpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=GEMINI_API_KEY,
    ),
)

## 3. Find recent news

In [75]:
from ddgs import DDGS


def search_news(query: str) -> str:
    """Find recent news and return titles, summaries, dates, and URLs."""
    results = DDGS(timeout=15).news(query, timelimit="d", max_results=10)
    if not results:
        raise RuntimeError("No recent news was found. Try a different topic.")

    for index, item in enumerate(results, start=1):
        print(f"{index}. {item.get('title', 'Untitled')}")
        print(item.get('url', 'No URL available.'))

    return "\n\n".join(
        f"Title: {item.get('title', 'Untitled')}\nDate: {item.get('date', 'Unknown')}\n"
        f"Summary: {item.get('body', 'No summary available.')}\nURL: {item.get('url', '')}"
        for item in results
    )


topic = "artificial intelligence business"
news_items = search_news(topic)

1. Meta (META)'s Stilla.ai Deal Could Turn Business Agent Into a New Monetization Engine
https://www.insidermonkey.com/news/meta-metas-stilla-ai-deal-could-turn-business-agent-into-a-new-monetization-engine-1837535/
2. The lucrative business of reactionary AI-generated posts on Facebook
https://www.lemonde.fr/en/pixels/article/2026/09/19/the-lucrative-business-of-reactionary-ai-generated-posts-on-facebook_6757721_13.html
3. The September 2026 AI Surge: Why Your Business Needs to Adapt Now
https://www.thetechedvocate.org/the-september-2026-ai-surge-why-your-business-needs-to-adapt-now/
4. Meta (META)’s Stilla.ai Deal Could Turn Business Agent Into a New Monetization E...
https://finance.yahoo.com/technology/ai/articles/meta-meta-stilla-ai-deal-075756581.html
5. Better Artificial Intelligence Stock: Aehr Test Systems vs. KLA Corporation
https://finance.yahoo.com/markets/stocks/articles/better-artificial-intelligence-stock-aehr-162316240.html
6. Google Gemini accessed protected systems of

## 4. Refine, check, and write the news

In [76]:
from agents import Agent, Runner

editor_agent = Agent(
    name="News Editor",
    instructions="Select three relevant, credible, non-duplicative items. Exclude clickbait, ads, speculation, and weak evidence. Preserve the key facts, dates, and URLs.",
    model=gemini_model,
)
fact_checker_agent = Agent(
    name="Fact Consistency Checker",
    instructions="Compare the edited news with the supplied source items. Remove or correct claims that are not supported by those items. Do not add new facts. Return a concise, source-grounded brief for the script writer.",
    model=gemini_model,
)
writer_agent = Agent(
    name="News Script Writer",
    instructions="Write a neutral 45- to 60-second spoken news script using only the supplied items. Do not add unsupported facts. End by naming the source publications without reading URLs aloud.",
    model=gemini_model,
)

editor_result = await Runner.run(editor_agent, f"Topic: {topic}\n\nNews items:\n{news_items}")
edited_news = editor_result.final_output
fact_check_result = await Runner.run(
    fact_checker_agent,
    f"Source items:\n{news_items}\n\nEdited news:\n{edited_news}",
)
fact_checked_news = fact_check_result.final_output
writer_result = await Runner.run(writer_agent, fact_checked_news)
news_script = writer_result.final_output
print(news_script)

On September 9th, 2026, Meta Platforms acquired the artificial intelligence startup Stilla.ai. This acquisition represents a step toward monetizing AI beyond Meta's core advertising business, potentially turning a business agent into a new monetization engine.

In technology security, Google Gemini successfully accessed the protected systems of three real companies during an AI cybersecurity test. In one case, the AI repeatedly guessed passwords to gain access.

Finally, Facebook pages based in Sri Lanka, Vietnam, and Venezuela are targeting French audiences with AI-generated political content. Often supportive of the far-right, these reactionary posts are being run as a lucrative business with profit as the primary goal.

Reporting for this broadcast comes from Insider Monkey, Yahoo Finance, Fox Business, and Le Monde.


## 5. Video Director Agent

The Video Director creates a structured Veo plan as JSON, then a Video Prompt Reviewer checks it for accuracy and suitability. A small function converts the approved plan into the text prompt and aspect ratio sent to Veo.

In [78]:
import json

VIDEO_PLAN_SCHEMA = '''{
  \"title\": \"short internal title\",
  \"duration_seconds\": 8,
  \"aspect_ratio\": \"16:9\",
  \"visual_style\": \"cinematic editorial news B-roll\",
  \"shots\": [{\"time\": \"0-3 seconds\", \"visual\": \"...\", \"camera\": \"...\"}],
  \"audio\": \"natural ambient sound only; no narration\",
  \"negative_constraints\": [\"no logos\", \"no text overlays\", \"no named real people\"]
}'''

video_director_agent = Agent(
    name="Video Director",
    instructions=(
        "Create a Veo plan for an 8-second, landscape, editorial news B-roll video. "
        "Use only visual themes supported by the supplied script. Do not invent claims or include logos, "
        "text overlays, named real people, or spoken narration. Return valid JSON only, with no Markdown, "
        "using exactly this schema:\n" + VIDEO_PLAN_SCHEMA
    ),
    model=gemini_model,
)

director_result = await Runner.run(video_director_agent, news_script)
veo_plan_json = director_result.final_output

video_plan_reviewer_agent = Agent(
    name="Video Plan Reviewer",
    instructions=(
        "Review the proposed Veo JSON plan against the supplied news script. Correct unsupported visual "
        "claims and ensure it has 2-3 shots, an 8-second duration, a 16:9 aspect ratio, no logos, text "
        "overlays, named real people, or spoken narration. Return the corrected valid JSON only, with no Markdown, "
        "using exactly this schema:\n" + VIDEO_PLAN_SCHEMA
    ),
    model=gemini_model,
)
review_result = await Runner.run(
    video_plan_reviewer_agent,
    f"News script:\n{news_script}\n\nProposed Veo JSON plan:\n{veo_plan_json}",
)
reviewed_veo_plan_json = review_result.final_output


def extract_json_object(agent_output: str) -> str:
    """Extract a JSON object when a model wraps it in Markdown or explanatory text."""
    text = str(agent_output).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("The Video Plan Reviewer did not return a JSON object.")
    return text[start : end + 1]


def build_veo_request(plan_json: str) -> tuple[str, str, int]:
    """Validate the reviewed plan and convert it to Veo prompt and configuration values."""
    try:
        plan = json.loads(extract_json_object(plan_json))
    except json.JSONDecodeError as error:
        raise ValueError("The video plan must be valid JSON.") from error

    required_fields = {"title", "duration_seconds", "aspect_ratio", "visual_style", "shots", "audio", "negative_constraints"}
    missing_fields = required_fields - plan.keys()
    if missing_fields or not isinstance(plan["shots"], list) or not plan["shots"]:
        raise ValueError(f"The video plan is missing required fields: {sorted(missing_fields)}")

    if plan["duration_seconds"] not in {4, 6, 8}:
        raise ValueError("Veo duration_seconds must be 4, 6, or 8.")
    if plan["aspect_ratio"] not in {"16:9", "9:16"}:
        raise ValueError("Veo aspect_ratio must be 16:9 or 9:16.")

    shot_text = "\n".join(
        f"{shot.get('time', 'Sequence')}: {shot.get('visual', '')}. Camera: {shot.get('camera', '')}."
        for shot in plan["shots"]
    )
    prompt = (
        f"Create an {plan['duration_seconds']}-second {plan['visual_style']} video.\n"
        f"Title: {plan['title']}\nShots:\n{shot_text}\n"
        f"Audio: {plan['audio']}\nAvoid: {', '.join(plan['negative_constraints'])}."
    )
    return prompt, plan["aspect_ratio"], plan["duration_seconds"]


cleaned_veo_plan_json = extract_json_object(reviewed_veo_plan_json)
reviewed_video_prompt, video_aspect_ratio, video_duration_seconds = build_veo_request(cleaned_veo_plan_json)
print(json.dumps(json.loads(cleaned_veo_plan_json), indent=2))
print("\nVeo prompt:\n", reviewed_video_prompt)

{
  "title": "AI Technology and Security News B-roll",
  "duration_seconds": 8,
  "aspect_ratio": "16:9",
  "visual_style": "cinematic editorial news B-roll",
  "shots": [
    {
      "time": "0-4 seconds",
      "visual": "A close-up shot in a dimly lit cybersecurity center. A computer monitor displays a command-line interface with rapid, automated lines of code and shifting alphanumeric characters, simulating an AI system executing brute-force password attempts on a secure network.",
      "camera": "Slow, steady push-in with a shallow depth of field, focusing on the green and white text reflecting off the monitor screen."
    },
    {
      "time": "4-8 seconds",
      "visual": "An over-the-shoulder shot of a modern control room. A large digital wall map of the world lights up, showing bright, glowing data streams originating from South America and Southeast Asia, bridging across the ocean to land directly on a highlighted European country.",
      "camera": "Slow pan and tilt up, 

## 6. Generate the video with Veo

Veo runs as a long-running job. This cell sends the approved structured plan as a Veo prompt, waits for completion, saves `news_brief.mp4`, and displays the result.

In [79]:
import time

from google import genai
from google.genai import types
from IPython.display import Video, display

veo_client = genai.Client(api_key=GEMINI_API_KEY)
operation = veo_client.models.generate_videos(
    model=VIDEO_MODEL_NAME,
    prompt=reviewed_video_prompt,
    config=types.GenerateVideosConfig(
        aspect_ratio=video_aspect_ratio,
        duration_seconds=video_duration_seconds,
    ),
)

while not operation.done:
    print("Waiting for video generation to complete...")
    time.sleep(10)
    operation = veo_client.operations.get(operation)

if not operation.response or not operation.response.generated_videos:
    raise RuntimeError("Video generation did not return a video. Check Veo access and try again.")

video_path = "news_brief.mp4"
generated_video = operation.response.generated_videos[0]
veo_client.files.download(file=generated_video.video, destination=video_path)
display(Video(video_path, embed=True))

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

## 7. Download the MP4

In [ ]:
from google.colab import files

files.download(video_path)